# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive full credit on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**For half credit, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description and feel free to experiment as you see fit.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

In [1]:
import altair as alt
import pandas as pd
import numpy as np

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [2]:
def legislators_df():
    df = pd.read_csv("data/legislators.csv")
    df["state"] = df["jurisdiction"].str.extract(r"(?:state|district|territory):(\w{2})/")[0]
    return df

# render dataframe
legislators_df()

,name,given_name,family_name,gender,party,jurisdiction,district,type,state
0,Robert White,Robert C.,White,Male,Independent,ocd-jurisdiction/country:us/district:dc/govern...,At-Large,legislature,dc
1,Charles Allen,Charles,Allen,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 6,legislature,dc
2,Vince Gray,Vincent Condol,Gray,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 7,legislature,dc
3,Phil Mendelson,Philip Heath,Mendelson,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Chairman,legislature,dc
4,Trayon White,Trayon,White,Male,Democratic,ocd-jurisdiction/country:us/district:dc/govern...,Ward 8,legislature,dc
...,...,...,...,...,...,...,...,...,...
7430,Sol Higgins,Sol Yamiz,Higgins Cuadrado,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,35,lower,pr
7431,Migdalia González,Migdalia I.,Gonzalez Arroyo,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,4,upper,pr
7432,Keren Riquelme,Keren L.,Riquelme Cabrera,Female,Partido Nuevo Progresista,ocd-jurisdiction/country:us/territory:pr/gover...,At-Large,upper,pr
7433,Jocelyne Rodríguez Negrón,Jocelyne M.,Rodriguez Negron,Female,Partido Popular Democrático,ocd-jurisdiction/country:us/territory:pr/gover...,19,lower,pr


### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [3]:
def states_by_gender_initial(df):
    return alt.Chart(df).mark_bar().encode(
        x = "state",
        y = "count()",
        color = "gender",
    )

# render chart
states_by_gender_initial(legislators_df())

alt.Chart(...)

### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [4]:
def states_by_gender_improved(df):
    bars = alt.Chart(df).mark_bar().encode(
        x = "state",
        y = alt.Y("count()").stack("normalize").axis(format="%").title("Share of legislators"),
        color = alt.Color("gender").scale(
            domain = ["Female", "Male", "Other"],
            range = ["#8624f5", "#1fc3aa", "gray"],
        ),
        order = alt.Order("gender", sort="ascending"),
    )

    line = alt.Chart(pd.DataFrame({"y": [0.5]})).mark_rule(color="gray").encode(
        y=alt.Y("y").title("Share of legislators")
    )
    return bars + line

# render chart
states_by_gender_improved(legislators_df())

alt.LayerChart(...)

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [5]:
def party_control_raw(df):
    bars = alt.Chart(df).mark_bar().encode(
        x = "state",
        y = alt.Y("count()").stack("normalize").axis(format="%").title("Share of legislators"),
        color = "party",
    )

    line = alt.Chart(pd.DataFrame({"y": [0.5]})).mark_rule(color="gray").encode(
        y = alt.Y("y").title("Share of legislators")
    )
    return bars + line

party_control_raw(legislators_df())

alt.LayerChart(...)

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [6]:
def clean_party_df():
    # start with the DataFrame from part 1 & return transformed copy
    df = legislators_df()
    df["party_code"] = "O"
    df.loc[df["party"].str.contains("Democratic"), "party_code"] = "D"
    df.loc[df["party"].str.contains("Republican"), "party_code"] = "R"
    df = df[~df["state"].isin(["dc", "ne", "pr"])]
    return df

clean_party_df()

,name,given_name,family_name,gender,party,jurisdiction,district,type,state,party_code
13,Kevin McCabe,Kevin J.,McCabe,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,30,lower,ak,R
14,C.J. McCormick,Conrad J.,McCormick,Male,Democratic,ocd-jurisdiction/country:us/state:ak/government,38,lower,ak,D
15,Bill Wielechowski,Bill P.,Wielechowski,Male,Democratic,ocd-jurisdiction/country:us/state:ak/government,K,upper,ak,D
16,Justin Ruffridge,Justin,Ruffridge,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,7,lower,ak,R
17,Will Stapp,Will,Stapp,Male,Republican,ocd-jurisdiction/country:us/state:ak/government,32,lower,ak,R
...,...,...,...,...,...,...,...,...,...,...
7353,Karlee Provenza,Karlee R.,Provenza,Female,Democratic,ocd-jurisdiction/country:us/state:wy/government,45,lower,wy,D
7354,Affie Ellis,Affie Burnside,Ellis,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,8,upper,wy,R
7355,Abby Angelos,Abby,Angelos,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,3,lower,wy,R
7356,Pepper Ottman,Pepper L.,Ottman,Female,Republican,ocd-jurisdiction/country:us/state:wy/government,34,lower,wy,R


### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [7]:
def party_control_by_chamber(df):
    bars = alt.Chart().mark_bar().encode(
        y = "state",
        x = alt.X("count()").stack("normalize").axis(format="%").title("Share of legislators"),
        color = alt.Color("party_code").scale(
            domain = ["D", "O", "R"],
            range = ["lightblue", "lightgreen", "darkred"],
        ),
    )

    line = alt.Chart().mark_rule(color="gray").encode(
        x = alt.datum(0.5)
    )

    return alt.layer(bars, line, data=df).facet(column="type")

party_control_by_chamber(clean_party_df())

alt.FacetChart(...)

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [8]:
def population_combined_df():
    seats = legislators_df().groupby(["state", "type"]).size().unstack(fill_value=0)
    seats = seats[["upper", "lower"]].reset_index()
    pops = pd.read_csv("data/populations.csv", encoding="utf-8-sig")
    return seats.merge(pops, on="state")

population_combined_df()

,state,upper,lower,pop_2020
0,ak,20,40,733374
1,al,35,104,5024294
2,ar,35,98,3011490
3,az,30,59,7157902
4,ca,40,79,39538212
5,co,35,65,5773707
6,ct,36,151,3605912
7,dc,0,0,689548
8,de,21,41,989946
9,fl,40,119,21538216


### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [9]:
def scatter_pop_size():
    df = population_combined_df()
    base = alt.Chart(df)

    upper = base.mark_point(shape="triangle-up", color="purple").encode(
        x = alt.X("pop_2020").title("Population (2020)"),
        y = alt.Y("upper").title("Number of seats"),
    )
    lower = base.mark_point(shape="triangle-down", color="orange").encode(
        x = "pop_2020",
        y = "lower",
    )

    return upper + lower

scatter_pop_size()

alt.LayerChart(...)

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [10]:
def scatter_pop_size_regression():
    df = population_combined_df()
    base = alt.Chart(df)

    upper = base.mark_point(shape="triangle-up", color="purple").encode(
        x = alt.X("pop_2020").title("Population (2020)"),
        y = alt.Y("upper").title("Number of seats"),
    )
    lower = base.mark_point(shape="triangle-down", color="orange").encode(
        x = "pop_2020",
        y = "lower",
    )

    upper_line = upper.transform_regression("pop_2020", "upper").mark_line(color="purple")
    lower_line = lower.transform_regression("pop_2020", "lower").mark_line(color="orange")

    return upper + lower + upper_line + lower_line

scatter_pop_size_regression()

alt.LayerChart(...)

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [11]:
def actions_df():
    df = pd.read_csv("data/actions_il-in-mi-wi_2021-2024.csv")
    df["date"] = pd.to_datetime(df["date"].str[:10])
    return df

actions_df()

,description,date,classification,state,session
0,"Filed with the Clerk by Rep. Marcus C. Evans, Jr.",2021-02-10,['filing'],IL,102nd
1,Filed with Secretary,2021-08-26,['filing'],IL,102nd
2,Filed with Secretary by Sen. Jacqueline Y. Col...,2021-02-25,['filing'],IL,102nd
3,Filed with Secretary,2022-01-19,['filing'],IL,102nd
4,Filed with Secretary,2022-11-29,['filing'],IL,102nd
...,...,...,...,...,...
456400,Partial veto failed to pass notwithstanding th...,2022-05-17,[],WI,2021
456401,Fiscal estimate received,2021-02-08,['receipt'],WI,2021
456402,Fiscal estimate received,2021-02-10,['receipt'],WI,2021
456403,Placed on calendar 5-17-2022 pursuant to Joint...,2022-05-17,[],WI,2021


### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [12]:
def actions_heatmap_scaled(df):
    daily = df.groupby(["state", "date"]).size().reset_index(name="actions")
    return alt.Chart(daily).mark_rect().encode(
        x = alt.X("yearweek(date):T").title("date (year-week)"),
        y = "state",
        color = alt.Color("sum(actions):Q").scale(scheme="yellowgreenblue").title("actions"),
    )

actions_heatmap_scaled(actions_df())

alt.Chart(...)

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [13]:
actions_heatmap_scaled(actions_df()[actions_df()["state"] != "IL"])

alt.Chart(...)

In [14]:
actions_heatmap_scaled(actions_df()[actions_df()["state"] == "IL"])

alt.Chart(...)

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [15]:
def actions_cumulative(df):
    daily = df.groupby(["state", "date"]).size().reset_index(name="actions")
    return alt.Chart(daily).transform_window(
        cumulative_actions = "sum(actions)",
        sort = [alt.SortField("date")],
        groupby = ["state"],
    ).mark_line().encode(
        x = "date:T",
        y = "cumulative_actions:Q",
        color = "state",
    )

actions_cumulative(actions_df())

alt.Chart(...)

## Part 5. Replicating a Chart

Find a chart (perhaps the one you're using for your first Analysis post!) and do your best to replicate it with Altair. Since you likely won't have access to the raw data-- start by mocking up a dataset that is similar enough to what your chart uses, don't worry about exact values, just that the correct attributes are present with some variation.

Your replicated chart should use similar colors, labels, etc. to the one that you chose to reproduce. Again, we aren't looking for a pixel-perfect replica, but something that pushes you to explore parts of Altair's documentation/features that you didn't use above.


**Original figure** (Kozlowski, Taddy & Evans 2019, *The Geometry of Culture*, Figure 3):

![Original Figure 3](imgs/ex5_fig3_original.png)

**My replication**

I'm replicating figure 3 from Kozlowski, Taddy and Evans (2019), The Geometry of Culture, American Sociological Review. The data is mock, as instructed, with positions read from the published figure.

In [16]:
# Mock data: positions are read by eye from the published figure.
genres = pd.DataFrame({
    "genre":        ["opera", "jazz", "bluegrass", "punk", "techno", "hiphop", "rap"],
    "survey_race":  [20, 63, 28, 25, 32, 82, 83],
    "survey_class": [84, 63, 32, 28, 41, 27, 22],
    "proj_race":    [-0.02, 0.15, 0.025, 0.01, 0.057, 0.21, 0.23],  # opera sits slightly left of 0.00 in the original
    "proj_class":   [0.06, 0.10, 0.013, -0.008, 0.02, -0.048, -0.075],
})

# The projection values are tiny (-0.1 to 0.2) and the survey values run 0-100.
# Convert projection -> survey units so every mark can share one scale; the top/right axes get relabeled later.
X_OFF, X_SCALE = 14, 360   # proj_race 0.00 sits at x=14, and 0.1 of projection = 36 survey units
Y_OFF, Y_SCALE = 50, 360   # proj_class 0.00 sits at y=50
W, H = 480, 420            # chart size is fixed here so the arrow angles below can be computed in pixels

genres["proj_x"] = X_OFF + X_SCALE * genres["proj_race"]
genres["proj_y"] = Y_OFF + Y_SCALE * genres["proj_class"]

# Direction of each survey -> projection arrow, in degrees clockwise from "up".
# dx and dy are scaled by the chart's pixel width and height, so the arrowhead points the right way on screen.
genres["angle"] = np.degrees(np.arctan2((genres["proj_x"] - genres["survey_race"]) * W,
                                        (genres["proj_y"] - genres["survey_class"]) * H))


In [17]:
TAUPE = "#b5a79d"  # the original's warm gray for everything on the projection side

def figure3_replica():
    xs = alt.Scale(domain=[0, 100], nice=False)   # nice=False keeps the axes ending exactly at 0 and 100
    ys = alt.Scale(domain=[0, 100], nice=False)
    base = alt.Chart(genres)

    # Layer 1: the link from each survey position to its projection position
    links = base.mark_rule(color=TAUPE, strokeWidth=2, opacity=0.35).encode(
        x = alt.X("survey_race:Q", scale=xs, axis=None),
        y = alt.Y("survey_class:Q", scale=ys, axis=None),
        x2 = "proj_x:Q",
        y2 = "proj_y:Q",
    )

    # Layer 2: a small triangle at the projection end, rotated by the angle column, to make arrows
    heads = base.mark_point(shape="triangle", filled=True, size=70, color=TAUPE, opacity=0.5).encode(
        x = alt.X("proj_x:Q", scale=xs, axis=None),
        y = alt.Y("proj_y:Q", scale=ys, axis=None),
        angle = alt.Angle("angle:Q").scale(None),   # scale(None): use the angle values as degrees as-is
    )

    # Layer 3: projection labels and the top/right axes
    # taupe axes and titles, with end labels "white/black" and "poor/rich"
    projection = base.mark_text(color=TAUPE, fontSize=13).encode(
        x = alt.X("proj_x:Q", scale=xs).title("Race Projection").axis(
            orient="top", values=[0, 14, 50, 86, 100], grid=False,
            labelExpr=f"datum.value == 0 ? 'white' : datum.value == 100 ? 'black' : format((datum.value - {X_OFF}) / {X_SCALE}, '.2f')",
            titleColor=TAUPE, labelColor=TAUPE, tickColor=TAUPE, domainColor=TAUPE,
            titleFontSize=18, labelFontSize=11),
        y = alt.Y("proj_y:Q", scale=ys).title("Class Projection").axis(
            orient="right", values=[0, 14, 50, 86, 100], grid=False,
            labelExpr=f"datum.value == 0 ? 'poor' : datum.value == 100 ? 'rich' : format((datum.value - {Y_OFF}) / {Y_SCALE}, '.2f')",
            titleColor=TAUPE, labelColor=TAUPE, tickColor=TAUPE, domainColor=TAUPE,
            titleFontSize=18, labelFontSize=11),
        text = "genre",
    )

    # Layer 4: survey labels and the bottom/left axes (drawn last so the black labels sit on top)
    # ticks every 25, end labels, faint gridlines
    survey = base.mark_text(color="black", fontWeight="bold", fontSize=13).encode(
        x = alt.X("survey_race:Q", scale=xs).title("Race Survey").axis(
            values=[0, 25, 50, 75, 100], grid=True, gridColor="#f1ece6",
            labelExpr="datum.value == 0 ? 'white' : datum.value == 100 ? 'black' : datum.value",
            titleFontSize=18, labelFontSize=12),
        y = alt.Y("survey_class:Q", scale=ys).title("Class Survey").axis(
            values=[0, 25, 50, 75, 100], grid=True, gridColor="#f1ece6",
            labelExpr="datum.value == 0 ? 'poor' : datum.value == 100 ? 'rich' : datum.value",
            titleFontSize=18, labelFontSize=12),
        text = "genre",
    )

    return (alt.layer(links, heads, projection, survey)
            .resolve_axis(x="independent", y="independent")   # keep the two sets of axes separate
            .properties(width=W, height=H)
            .configure_view(stroke=None))                      # remove the box around the plot area

figure3_replica()

alt.LayerChart(...)

The figure above is my final version. Below is my first attempt (version 1), kept to show how I revised it. Comparing it with the original, I changed:

| | Version 1 | Final version |
|---|---|---|
| Links | plain light-gray lines | faint taupe lines with arrowheads (`mark_point` + `alt.Angle`) |
| Projection colors | neutral gray | the original's warm taupe on labels, axes and titles |
| Axis ticks | default ticks, top/right axes showed only three values | ticks every 25, plus 0.00 / 0.10 / 0.20 and -0.10 / 0.00 / 0.10 |
| End labels | none | "white / black" and "poor / rich" via `labelExpr` |
| Grid and frame | default | faint gridlines, no border (`configure_view`) |
| Data | rough positions, magic numbers in the conversion | positions re-read from the original (opera moved left); named constants |

**Version 1** (its own data and function name, so it doesn't clash with the final version):

In [18]:
# Version 1: my first attempt, kept to show the progress.
# It has its own data (genres_v1) and function name so it doesn't clash with the final version above.
genres_v1 = pd.DataFrame({
    "genre":        ["opera", "jazz", "bluegrass", "punk", "techno", "hiphop", "rap"],
    "survey_race":  [19, 65, 32, 30, 34, 82, 83],
    "survey_class": [85, 65, 34, 30, 43, 28, 23],
    "proj_race":    [0.01, 0.15, 0.02, 0.01, 0.05, 0.20, 0.22],
    "proj_class":   [0.06, 0.10, 0.02, -0.01, 0.02, -0.04, -0.05],
})
genres_v1["proj_x"] = 15 + 400 * genres_v1["proj_race"]
genres_v1["proj_y"] = 50 + 400 * genres_v1["proj_class"]


def figure3_replica_v1():
    xs = alt.Scale(domain=[0, 100])
    ys = alt.Scale(domain=[0, 100])
    base = alt.Chart(genres_v1)

    links = base.mark_rule(color="#dddddd", strokeWidth=2).encode(
        x = alt.X("survey_race:Q", scale=xs, axis=None),
        y = alt.Y("survey_class:Q", scale=ys, axis=None),
        x2 = "proj_x:Q",
        y2 = "proj_y:Q",
    )
    survey = base.mark_text(color="black", fontWeight="bold", dy=-8).encode(
        x = alt.X("survey_race:Q", scale=xs).title("Race Survey"),
        y = alt.Y("survey_class:Q", scale=ys).title("Class Survey"),
        text = "genre",
    )
    projection = base.mark_text(color="#bbbbbb", dy=-8).encode(
        x = alt.X("proj_x:Q", scale=xs).title("Race Projection").axis(
            orient="top", values=[15, 55, 95], labelExpr="format((datum.value - 15) / 400, '.2f')"),
        y = alt.Y("proj_y:Q", scale=ys).title("Class Projection").axis(
            orient="right", values=[10, 50, 90], labelExpr="format((datum.value - 50) / 400, '.2f')"),
        text = "genre",
    )
    return alt.layer(links, projection, survey).resolve_axis(
        x="independent", y="independent"
    ).properties(width=420, height=360)


figure3_replica_v1()

alt.LayerChart(...)